<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Konzepte &amp; Embeddings
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M11-RAG-Konzepte"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 34.58.107.95
Hostname: 95.107.58.34.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


In [2]:
#@title 📦 Pakete installieren{ display-mode: "form" }

# install_packages([
#                 ('markitdown[all]', 'markitdown'),
#                 'langchain_huggingface',
#                 ('unstructured[all-docs]>=0.11.2', 'unstructured'),
#                 ])

# 1 | Übersicht
---



**RAG** (**R**etrieval-**A**ugmented **G**eneration) erweitert Sprachmodelle um **externes Wissen**.  
Ein Agent kann damit Informationen abrufen, die nicht im Training enthalten waren –  
z. B. aktuelle Daten, firmeneigene Dokumente oder spezialisiertes Fachwissen.




<p><font color='black' size="5">
Vergleich: LLM vs. LLM + RAG
</font></p>

| Eigenschaft | LLM allein | LLM + RAG |
|-------------|-----------|----------|
| Wissensbasis | Bis Trainings-Cutoff | Ständig aktualisierbar |
| Private Daten | ❌ Nicht vorhanden | ✅ Aus Vektordatenbank |
| Halluzinationen | Häufig bei unbekanntem Wissen | Reduziert durch Quellenbindung |
| Anpassbarkeit | Fine-Tuning notwendig | Neue Dokumente einfach hinzufügen |
| Transparenz | Schwer nachvollziehbar | Quellen sichtbar |



# 2 | Warum RAG?
---

<p><font color='black' size="5">
Das Problem: LLMs sind eingefroren
</font></p>





Sprachmodelle werden mit Daten bis zu einem bestimmten Zeitpunkt trainiert.  
Danach wissen sie nichts über:

- 📅 **Neue Ereignisse** – Aktuelles aus Wirtschaft, Technik, Wissenschaft
- 🔒 **Vertrauliche Daten** – Interne Dokumente, Kundendaten, proprietäres Wissen
- 📚 **Spezialisiertes Fachwissen** – Produktdokumentation, Rechtsnormen, Handbücher



<p><font color='black' size="5">
Die RAG-Lösung
</font></p>




RAG trennt das **Reasoning** (im LLM) vom **Wissen** (in der Datenbank):

1. 🔍 **Retrieval** – Relevante Dokumente aus einer Wissensbasis abrufen
2. 🔧 **Augmentation** – Den LLM-Prompt mit diesen Dokumenten anreichern
3. ✍️ **Generation** – Das LLM antwortet basierend auf den abgerufenen Informationen

**Wann RAG verwenden?**

| Szenario | RAG? | Begründung |
|----------|------|------------|
| FAQ für eigene Produkte | ✅ | Private, sich ändernde Daten |
| Allgemeine Wissensfragen | ❌ | LLM-Basiswissen genügt |
| Interne Dokumentensuche | ✅ | Vertrauliche Unternehmensdaten |
| Mathematik / Code | ❌ | Tools sind effizienter |
| Aktuelle Nachrichten | ✅ | Übersteigt Trainings-Cutoff |

# 3 | RAG-Architektur
---


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process.png" width="700" alt="Avatar">

*RAG-Prozess: Von der Indexierung bis zur Antwort-Generierung*

<p><font color='black' size="5">
Datensammlung
</font></p>

+ Dokumente:
Relevante Texte (z.B. Fachartikel, Handbücher, Webseiten) werden gesammelt.

+ Chunking: Aufteilung der längere Texte in kleinere, zusammenhängende Abschnitte (Chunks).

+ Embedding:
Die Texte werden mithilfe eines Embedding-Modells (z. B. sentence-transformers) in numerische Vektoren umgewandelt.

+ Vector Database:
Die Embeddings und zugehörigen Texte werden in einer Vektordatenbank (z. B. FAISS, ChromaDB) gespeichert, um später effizient durchsucht werden zu können.

<p><font color='black' size="5">
Abruf & Erweiterung
</font></p>




+ User Query:
Der Nutzer stellt eine Anfrage (z. B. eine Frage in natürlicher Sprache).

+ Query Embedding:
Die Frage wird in ein Embedding konvertiert

+ Retriever:
Ein Suchsystem durchsucht eine (externe) Vektordatenbank (z.B. mit Dokumenten) nach relevanten Inhalten zur Anfrage.

+ Dokumente:
Die gefundenen, relevanten Dokumente (Kontextinformationen) werden gesammelt.


+ Promptanreicherung:
Die Abfrage des Nutzers wird um die gefundenen Dokumente ergänzt.

+ Model Inference:
Das generative Sprachmodell (z.B. GPT) erhält die ursprüngliche Anfrage plus die gefundenen Dokumente als Kontext. Es generiert auf dieser Grundlage eine fundierte Antwort.

[Bild Kundenanfrage](https://www.researchgate.net/publication/378467192/figure/fig3/AS:11431281235857152@1712887907223/Zusammenspiel-zwischen-Vektordatenbank-Kundenanfrage-und-LLM.png)


Quelle: [Die_Nutzung_von_ChatGPT_in_Unternehmen](https://www.researchgate.net/publication/378467192_Die_Nutzung_von_ChatGPT_in_Unternehmen_Ein_Fallbeispiel_zur_Neugestaltung_von_Serviceprozessen)


<p><font color='black' size="5">
Einschränkungen von RAG
</font></p>



LLM RAG kombiniert Sprachmodelle mit externem Datenabruf, um fehlendes Wissen aus spezialisierten oder proprietären Quellen zu ergänzen. Dies ist besonders nützlich in Bereichen wie Finanzen, Recht oder Technik, wo präzise und aktuelle Informationen erforderlich sind.

Wenn die zusätzlichen Daten jedoch bereits Allgemeinwissen sind, bringt RAG wenig Mehrwert. Da das Basismodell umfassend vortrainiert ist, kann es viele Anfragen ohne externe Erweiterung beantworten. Der unnötige Abruf bekannter Informationen führt zudem zu Ineffizienzen und erhöht den Rechenaufwand.

RAG sollte gezielt eingesetzt werden, insbesondere für domänenspezifische oder proprietäre Daten, die das Basismodell nicht abdeckt. In allgemeinen Wissensbereichen ist es meist überflüssig.

# 4 | Token-Limits & Context Window
---

Große Sprachmodelle (LLMs) wie GPT-3 oder BERT verarbeiten Text nicht als ganze Sätze oder Absätze, sondern als eine Folge von "Tokens". Tokenisierung und Chunking sind entscheidende Vorverarbeitungsschritte, die es ermöglichen, Texte effizient zu verarbeiten und die Leistung von KI-Modellen zu optimieren.

**Tokenisierung: Die Grundlage der Textverarbeitung**

Tokenisierung ist der Prozess, bei dem Text in kleinere Einheiten, sogenannte Tokens, zerlegt wird. Diese Tokens können Wörter, Teilwörter oder sogar einzelne Zeichen sein.

**Warum ist Tokenisierung wichtig?**

1. **Einheitliche Verarbeitung:** LLMs haben eine begrenzte Eingabelänge. Die Tokenisierung stellt sicher, dass Texte in einem einheitlichen Format vorliegen, das vom Modell verarbeitet werden kann.

2. **Bedeutungserfassung:** Viele Tokens repräsentieren semantische Einheiten, was dem Modell hilft, die Bedeutung des Textes besser zu erfassen.

3. **Effiziente Verarbeitung:** Tokenisierte Texte können effizienter verarbeitet und in numerische Vektoren umgewandelt werden, was für die Eingabe in neuronale Netze notwendig ist.


**Hilfe:**

1 DIN A4 Seite hat ca. 300 Worte und ca. 450 Token

[OpenAI Tokenizer](https://platform.openai.com/tokenizer)

Das Kontextfenster muss für **drei Komponenten** reichen:

```
Context Window (z.B. 128k Tokens bei GPT-4o)
┌──────────────┬──────────────────┬──────────────┐
│ System-Prompt│  Top-K Chunks    │  Antwort     │
│  ~300 Tokens │  K x 400 Tokens  │  ~500 Tokens │
└──────────────┴──────────────────┴──────────────┘
```

**Praxisregeln**

| Parameter | Empfehlung | Begründung |
|-----------|-----------|------------|
| `chunk_size` | 300–500 Zeichen | Kompakte, fokussierte Chunks |
| `chunk_overlap` | 10–15% von chunk_size | Verhindert Informationsverlust an Grenzen |
| Top-K | 3–5 Chunks | Meist ausreichend, spart Tokens |
| Embedding-Modell | `text-embedding-3-small` | Gutes Preis-Leistungs-Verhältnis |

- Warum Chunk-Größe und Overlap Kosten und Qualität beeinflussen
- Prompt-Budget: Frage + Kontext + Antwortfenster vorausplanen
- Retrieval-Tiefe nur so groß wie nötig

# 5 | Chunking-Strategien
---

<p><font color='black' size="5">
Warum Chunking?
</font></p>

**Chunking: Texte in verdauliche Häppchen teilen**

Chunking ist der Prozess, bei dem längere Texte in kleinere, zusammenhängende Abschnitte (Chunks) aufgeteilt werden.

**Warum ist Chunking wichtig?**

1. **Verarbeitung langer Texte:** Da LLMs eine begrenzte Eingabelänge haben, ermöglicht Chunking die Verarbeitung von Texten, die länger als diese Grenze sind.

2. **Kontexterhaltung:** Durch geschicktes Chunking kann der relevante Kontext innerhalb eines Chunks erhalten bleiben, was für viele NLP-Aufgaben entscheidend ist.

3. **Effizienz:** Kleinere Chunks können parallel verarbeitet werden, was die Gesamtverarbeitungszeit reduzieren kann.

**Strategien im Vergleich**

| Strategie | Beschreibung | Vorteil | Nachteil |
|-----------|-------------|---------|----------|
| **Fixed-Size** | Feste Zeichenanzahl | Einfach, schnell | Zerreißt Sätze |
| **Sentence-based** | An Satzgrenzen | Semantisch sauber | Variable Größe |
| **Recursive** | Hierarchisch: Absatz → Satz → Wort | ✅ Ausgewogen | Komplexer |
| **Semantic** | Nach semantischen Gruppen | Beste Qualität | Langsam, teuer |

**Empfehlung:**    
`RecursiveCharacterTextSplitter` mit `chunk_size=500, chunk_overlap=50`

> **Faustregel:** 1 DIN-A4-Seite ≈ 300 Wörter ≈ 450 Tokens

[ChunkViz](https://chunkviz.up.railway.app/)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "Kuenstliche Intelligenz veraendert die Arbeitswelt grundlegend. "
    "Viele Routineaufgaben werden automatisiert, waehrend kreative Taetigkeiten wichtiger werden.\n\n"
    "Agenten-Systeme sind besonders leistungsstark, weil sie eigenstaendig Entscheidungen treffen koennen. "
    "Ein Agent kann Tools aufrufen, Informationen abrufen und komplexe Aufgaben zerlegen.\n\n"
    "RAG-Systeme ergaenzen diese Faehigkeiten um externes Wissen. "
    "Durch Agenten und RAG entstehen Systeme, die sowohl denken als auch nachschlagen koennen."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len,
)

chunks = splitter.split_text(text)

print(f"Original:  {len(text)} Zeichen")
print(f"Chunks:    {len(chunks)} Stueck (chunk_size=200, overlap=30)\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} Zeichen) ---")
    print(chunk)
    print()

Original:  496 Zeichen
Chunks:    3 Stueck (chunk_size=200, overlap=30)

--- Chunk 1 (156 Zeichen) ---
Kuenstliche Intelligenz veraendert die Arbeitswelt grundlegend. Viele Routineaufgaben werden automatisiert, waehrend kreative Taetigkeiten wichtiger werden.

--- Chunk 2 (186 Zeichen) ---
Agenten-Systeme sind besonders leistungsstark, weil sie eigenstaendig Entscheidungen treffen koennen. Ein Agent kann Tools aufrufen, Informationen abrufen und komplexe Aufgaben zerlegen.

--- Chunk 3 (150 Zeichen) ---
RAG-Systeme ergaenzen diese Faehigkeiten um externes Wissen. Durch Agenten und RAG entstehen Systeme, die sowohl denken als auch nachschlagen koennen.



# 6 | Embeddings erklärt
---


<p><font color='black' size="5">
Was sind Embeddings?
</font></p>


Ein **Embedding** ist eine numerische Darstellung von Text als Vektor in einem hochdimensionalen Raum.  
Semantisch ähnliche Texte liegen dabei **nah beieinander** im Vektorraum.

```
"Hund spielt"  →  [0.23, -0.11,  0.87, ...]   <- nah beieinander!
"Tier tobt"    →  [0.21, -0.09,  0.84, ...]
"Aktienkurs"   →  [-0.45, 0.72, -0.31, ...]   <- weit entfernt
```



Einbettungen stellen eine KI-optimierte Repräsentation verschiedener Datentypen dar und eignen sich daher besonders für den Einsatz in einer Vielzahl KI-gestützter Tools und Algorithmen. Sie erfassen die wesentlichen Merkmale von Texten, Bildern und auch von Audio- und Videodaten.

Ein Einbettungsmodell verarbeitet Eingangsdaten und wandelt sie in numerische Vektoren um. Die Architektur des Modells sorgt dafür, dass ähnliche Inhalte – beispielsweise Texte mit verwandter Bedeutung oder visuell ähnliche Bilder – im Vektorraum näher beieinander liegen, während sich unähnliche Daten weiter voneinander entfernt befinden.



Ein **Embedding** ist ein Vektor aus Gleitkommazahlen, der Ähnlichkeiten zwischen Texten misst. Kleinere Abstände zwischen Vektoren bedeuten eine stärkere inhaltliche Nähe.

[Beispiel Fahrzeug 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)

[Beispiel Fahrzeug 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)



OpenAI bietet zwei Einbettungsmodelle an:
- **text-embedding-3-small**
- **text-embedding-3-large**

**Unterschiede und Auswahlkriterien:**

| Kriterium         | text-embedding-3-small | text-embedding-3-large |
|------------------|----------------------|----------------------|
| **Genauigkeit & Leistung** | Weniger detailliert, aber für viele Aufgaben ausreichend | Erfasst komplexere Zusammenhänge, ideal für anspruchsvolle NLP-Aufgaben |
| **Rechenaufwand** | Effizient, benötigt weniger Ressourcen | Höherer Speicher- und Rechenbedarf |
| **Latenz & Geschwindigkeit** | Schnellere Verarbeitung, geringe Latenz | Höhere Latenz, nicht ideal für Echtzeit-Anwendungen |
| **Kosten** | Kostengünstiger, ideal für skalierbare Anwendungen | Teurer in Betrieb und Bereitstellung |
| **Anwendungsfälle** | Chatbots, Echtzeitsysteme, einfache Textverarbeitung | Semantische Analyse, komplexe NLP-Aufgaben, KI-Forschung |



**Fazit**:  
- **Large**: Wenn hohe Präzision, detailliertes Sprachverständnis und ausreichend Ressourcen vorhanden sind.  
- **Small**: Wenn Effizienz, niedrige Kosten und schnelle Reaktionszeiten im Vordergrund stehen.  

Die Wahl des Modells hängt von den spezifischen Anforderungen der Anwendung ab.

<p><font color='black' size="5">
OpenAI Embedding-Modelle
</font></p>


| Modell | Dimensionen | Kosten | Empfehlung |
|--------|------------|--------|------------|
| `text-embedding-3-small` | 1.536 | Günstig | ✅ Standard |
| `text-embedding-3-large` | 3.072 | Höher | Anspruchsvolle Suche |
| `text-embedding-ada-002` | 1.536 | Mittel | ⚠️ Veraltet |



<p><font color='black' size="5">
Ähnlichkeitsmaße -  Vektoren vergleichen
</font></p>

In der Mathematik gibt es verschiedene Methoden, um Vektoren miteinander zu vergleichen. Einige der gängigsten Ansätze sind:


* **Kosinus-Ähnlichkeit**: Berechnet den Kosinus des Winkels zwischen zwei Vektoren, wobei die Orientierung stärker gewichtet wird als die Größe. Diese Technik ist besonders verbreitet in Bereichen wie Text Mining und der Informationssuche, um Dokumente auf Ähnlichkeit zu prüfen.
* **Euklidische Distanz**: Gibt die direkte Entfernung zwischen zwei Vektoren an und wird häufig in Clustering-Methoden oder Algorithmen zum nächsten Nachbarn im maschinellen Lernen verwendet.

* **Manhattan-Distanz**: Addiert die absoluten Differenzen der Komponenten zweier Vektoren. Diese Methode wird vor allem in gitterbasierten Pfadsuchalgorithmen sowie in bestimmten maschinellen Lernanwendungen eingesetzt.

Laut einer Empfehlung von OpenAI [FAQ](https://platform.openai.com/docs/guides/embeddings/frequently-asked-questions), eignet sich die Kosinus-Ähnlichkeit besonders gut zum Vergleich von Vektoren, da dabei die ursprüngliche Größe beibehalten wird. Zudem bleibt das Vorzeichen der einzelnen Vektorkomponenten erhalten, was für die Analyse entscheidend ist.




| Maß | Wertebereich | Empfehlung |
|-----|------------|------------|
| **Kosinus-Ähnlichkeit** | -1 bis 1 | ✅ OpenAI empfohlen |
| Euklidischer Abstand | 0 bis ∞ | Alternativ |

**Interpretation:** 1.0 = identisch  •  0.5 = verwandt  •  0.0 = unrelated  •  -1.0 = gegensätzlich

# 7 | Embedding-Demo
---

Wir erstellen Embeddings für drei Sätze und vergleichen ihre semantische Ähnlichkeit:

- **Satz A & B** – Beide über Tiere → sollten **ähnlich** sein
- **Satz A & C** – Tier vs. Börse → sollten **unähnlich** sein

In [4]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
print("✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)")

✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)


In [5]:
# Drei Beispielsätze
satz_a = "Der Hund spielt im Park."
satz_b = "Das Tier tobt auf der Wiese."
satz_c = "Die Aktie fiel gestern um 3 Prozent."

vektoren = embeddings_model.embed_documents([satz_a, satz_b, satz_c])

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sim_ab = cosine_similarity(vektoren[0], vektoren[1])
sim_ac = cosine_similarity(vektoren[0], vektoren[2])

mprint(f"""
## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen | {sim_ab:10.4f} | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse | {sim_ac:10.4f} | 🔴 Unähnlich |

Vektordimensionen: {len(vektoren[0])}
""")


## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen |     0.5439 | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse |     0.2002 | 🔴 Unähnlich |

Vektordimensionen: 1536


# 8 | Mini-RAG: Vom Chunk zur Antwort
---



Die vorherigen Abschnitte haben Embeddings und Chunking einzeln beleuchtet.  
Jetzt kombinieren wir beides zu einem **vollständigen RAG-Loop** – kompakt, ohne persistente Datenbank.

**Ablauf:**
1. 🗃️ 4 Mini-Dokumente → In-Memory Vektordatenbank  
2. 🔍 Retriever findet die 2 ähnlichsten Chunks  
3. 📝 Prompt wird mit Kontext angereichert  
4. 🤖 LLM antwortet quellgebunden

> **Hinweis:** Das nächste Modul (*ChromaDB & Indexing*) baut dieses Muster aus – mit ChromaDB-Persistenz, Collections und Metadaten-Filtern.

In [6]:
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Mini-Wissensdatenbank (4 Sätze, in-memory – kein persist_directory)
mini_docs = [
    'RAG kombiniert Retrieval mit LLM-Generation: Das LLM antwortet basierend auf abgerufenen Dokumenten.',
    'Embeddings sind numerische Vektoren – semantisch ähnliche Texte liegen nah beieinander im Vektorraum.',
    'Chunking teilt lange Dokumente in kleinere Abschnitte, damit sie ins Kontextfenster passen.',
    'ChromaDB ist eine Vektordatenbank, die Embeddings persistent speichert und schnell durchsucht.',
]

vectorstore = Chroma.from_texts(mini_docs, embedding=embeddings_model)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 2})

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

llm = init_chat_model('openai:gpt-4o-mini', temperature=0.0)

rag_prompt = PromptTemplate.from_template(
    'Beantworte die Frage ausschliesslich auf Basis des Kontexts.\n\n'
    'Kontext:\n{context}\n\n'
    'Frage: {question}\n\n'
    'Antwort:'
)

mini_rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print('✅ Mini-RAG-Chain bereit (in-memory, 4 Docs)')

✅ Mini-RAG-Chain bereit (in-memory, 4 Docs)


In [7]:
fragen = [
    'Was ist RAG?',
    'Wozu braucht man Chunking?',
    'Was macht ChromaDB?',
]

for frage in fragen:
    antwort   = mini_rag_chain.invoke(frage)
    retrieved = retriever.invoke(frage)
    mprint(f'''
---
**Frage:** {frage}
**Quellen:** {len(retrieved)} Chunk(s) abgerufen
**Antwort:** {antwort}
''')


---
**Frage:** Was ist RAG?
**Quellen:** 2 Chunk(s) abgerufen
**Antwort:** RAG steht für Retrieval-Augmented Generation und kombiniert die Retrieval-Technik mit der Generierung durch ein Language Model (LLM). Das LLM antwortet dabei basierend auf abgerufenen Dokumenten.



---
**Frage:** Wozu braucht man Chunking?
**Quellen:** 2 Chunk(s) abgerufen
**Antwort:** Chunking wird benötigt, um lange Dokumente in kleinere Abschnitte zu unterteilen, damit sie ins Kontextfenster passen.



---
**Frage:** Was macht ChromaDB?
**Quellen:** 2 Chunk(s) abgerufen
**Antwort:** ChromaDB speichert Embeddings persistent und ermöglicht eine schnelle Durchsuchung dieser Vektordatenbank.


In [8]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M11-RAG-Konzepte", limit=3, show_steps=True)

## LangSmith Trace — `M11-RAG-Konzepte`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `RunnableSequence` | ❌ pending | — | 0 |
| `RunnableSequence` | ✅ success | 1.1s | 0 |
| `RunnableSequence` | ✅ success | 1.2s | 0 |


### Steps — letzter Run: `RunnableSequence`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `parser` | `StrOutputParser` | ✅ | 0.0s |
| 2 | `llm` | `ChatOpenAI` | ✅ | 0.9s |
| 3 | `prompt` | `PromptTemplate` | ✅ | 0.0s |
| 4 | `chain` | `RunnableParallel<context,question>` | ✅ | 1.1s |

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Embedding-Vergleich: Thematische Cluster
</font></p>

Erstellen Sie Embeddings für einen kleinen Textkorpus (6–9 Sätze aus 3 verschiedenen Themen)
und untersuchen Sie, ob die Ähnlichkeitsmaße die Themengruppen korrekt erkennen.

**Teilaufgaben:**
1. Wählen Sie 3 Themen (z.B. Sport, Technik, Kochen) mit je 2–3 Sätzen
2. Erstellen Sie die Embeddings mit `text-embedding-3-small`
3. Berechnen Sie die Kosinus-Ähnlichkeit für alle Paare
4. Stellen Sie die Ergebnisse in einer Markdown-Tabelle dar

**Bonus:** Vergleichen Sie `text-embedding-3-small` mit `text-embedding-3-large` –
gibt es Unterschiede bei der Clustererkennung?